<br><div align="center"><span style="font-size:200%;font-weight:bold">Introduction aux traitements automatisés du langage</span></div></br>

<div align="center"><span style="font-size:150%"> Arnold Vialfont, UPEC - ERUDITE, UNIL</span></div>

<br>

<div align="center"><span style="font-size:130%">Avril 2026</span></div>

***

L'objet de cette série de 3 notebooks est d'introduire les opérations classiques de la **fouille de texte** (*text mining*) et du **traitement automatique du langage naturel** (TALN ou *Natural Langage Processing* en anglais, NLP ci-après). Ces deux champs d'analyse visent à extraire de l'information au sens large à partir de **données non structurées** (des textes, sons, images ou vidéos). 

Le NLP traite spécifiquement des "**langues naturelles**", au sens de créations collectives, qui s'opposent a priori aux langues "artificielles" ou "formelles" (programmation informatique, mathématiques, etc). Le text mining inclut une partie du NLP dès lors que les textes sont en langage naturel (contrairement aux listings de mails, fichiers log, etc.). On se limite ici à présenter la partie du NLP sur données textuelles (sachant qu'un enregistrement audio ou vidéo peut être retranscrit en texte et un texte en son et/ou image). Pour une présentation complémentaire des méthodes de text mining 1, voir Rakotomalala (2017).

L'objectif principal de ce notebook et des deux suivants est de présenter comment il est possible d'extraire de l'information au sens large à partir de **données non structurées** que sont les textes : extraction d'éléments d'intérêt (expressions régulières, entités nommées, etc.), classification de documents, clustering, topic modeling, analyse de sentiments, etc. Pour cela nous aurons besoin d'évoquer les méthodes de l'apprentissage automatique (*machine learning*) et parfois, plus spécifiquement, celles de l'apprentissage profond (*deep learning*). La figure suivante présente les intersections de ces différents domaines.

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Les principaux domaines mobilisés dans ces notebooks</b><br>
  <img src="img/txtm_nlp_ml_dl.png" style="margin-top:0.5cm;margin-bottom:0cm;width:40%"> 
    </div>
</figure>

Le langage retenu dans les deux prochain notebooks est Python 3, en ayant le plus souvent recours à la librairie `SpaCy` qui est assez simple d'utilisation et compatible avec plusieurs langues (dont le français qui est souvent absent des packages historiques comme `nltk`). On a aussi recours aux pipelines issus de la librairie `Scikit-Learn` pour sérialiser les différentes opérations réalisées. La librairie `transformers` permet assez simplement d'accéder aux solutions de l'état de l'art (*SOTA* en anglais pour state-of-the-art) et sera présentée dans les dernier notebook. Pour l'exécution de ces notebooks, il est possible de les exécuter en passant par [le site Google Colab](https://colab.research.google.com/) (qui propose un environnement python flexible et de la puissance de calcul) : cliquer sur le lien "Open in Colab" en haut de la page.

Dans ce notebook d'introduction, aucune cellule n'est à exécuter. Nous décrivons simplement l'évantail des possibles du text mining et du NLP (section 1), puis nous présentons les deux principales approches de l'analyse du langage naturel (section 2). On présente ensuite la manière de passer d'un corpus à un objet d'étude statistique (section 3). Enfin, on voit les étapes classiques d'un projet de text mining, dont une phase d'évaluation des modèles construits (section 4).

# 1. Principales tâches et difficultés en text mining et NLP

Les tâches pouvant être réalisées avec le NLP sont notamment les suivantes :

- **Recherche à partir de mots-clés** : trouver des documents pertinents parmi un grand ensemble. Utilisée notamment par les moteurs de recherche.

- **Classification de textes** : on place des textes dans des catégories prédéfinies en fonction de leur contenu. Utilisé de la détection de spams à l'analyse de sentiments/opinions.

- **Clustering et modélisation de sujets** (topic modeling) : rapprocher des textes similaires et inférer les sujets contenus dans un grand ensemble de documents. Ces tâches sont très fréquente et peuvent être un préalable aux tâches de classification.

- **Extraction d'information** : recherche d'informations pertinentes dans un texte. Utile pour la recherche des "entités nommées" (noms propres, etc.) ou des évènements dans un calendrier ou dans un ensemble de textes, par exemple sur les réseaux sociaux.

- **modélisation du langage** : en vue de prédire le mot suivant dans un texte, on utilise l'historique des mots précédents et on construit la probabilité d'une séquence de mots à suivre dans un langage donné. Utilisée pour la reconnaissance optique des caractères manuscrits, la génération de texte, la correction orthographique avancée, etc.

- **Résumé de texte** : créer un résumé de longs documents tout en conservant le contenu principal et le sens global du texte. La [librairie Transformers](https://huggingface.co/tasks/summarization) propose de nombreux modèles pour cette tâche.

- **Modèle de question-réponse** : construction de systèmes qui peuvent répondre automatiquement aux questions posées en langage naturel. Parmi les solutions existantes, [celle développée par Etalab](https://www.data.gouv.fr/fr/reuses/modele-de-questions-reponses-francophone/) est une solution disponible pour le français.

- **Traduction automatique** : conversion d'un texte d'une langue vers une autre. Des outils comme [Google Translate](https://translate.google.fr/) ou [DeepL](https://www.deepl.com/translator) sont des exemples d'application de cette tâche.

- **Agent conversationnel** (*chatbot*) : système de dialogue permettant un échange ou l'envoi d'une commande en langage humain. Les versions les plus avancées de ces agents fonctionnent en "domaine ouvert" ([Alexa](https://developer.amazon.com/fr-FR/alexa), [Siri](https://www.apple.com/fr/siri/), [ChatGPT](https://chat.openai.com/), etc.).

La figure ci-dessous permet de refléter les degrés relatifs de difficulté des tâches évoquées.

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Les tâches du text mining et du NLP selon leur difficulté (Vajjala et al., 2020, p.8)</b> <br>
  <img src="img/tasks-NLP.png" style="margin-top:0.5cm;margin-bottom:0cm;width:55%"></div>
</figure>

Une des principales difficultés des méthodes de la fouille de texte au sens général est associée à [**l'ambiguïté**](https://fr.wikipedia.org/wiki/Ambigu%C3%AFt%C3%A9). Par exemple, certains mots peuvent avoir plusieurs sens, notamment selon le contexte du texte ou leur fonction grammaticale ("livre" peut être un objet ou un verbe, etc).

De plus, les textes ne contiennent pas l'intégralité des éléments sur lesquels reposent leur signification. Notamment, il y a généralement un **recours implicite au "sens commun"**. Par exemple, les phrases "*le chien a mordu le cycliste*" et  "*le cycliste a mordu le chien*" ne sont pas équiprobables : des informations non inclues dans le texte sont utiles pour le comprendre (ex: fake news ou non ?).

En pratique, il faudra aussi prendre en compte **la forme du texte** elle-même : encodage, balises, ponctuation et mise en page, variations du vocabulaire, de l'orthographe et de la grammaire utilisés, etc.

 RQ : le NLP évolue rapidement, et de nombreuses solutions sont proches de passer le [test de Turing](https://fr.wikipedia.org/wiki/Test_de_Turing), notamment [ChatGPT4 qui a su utiliser la ruse dans l'objectif de passer pour un humain](https://www.journaldugeek.com/2023/03/16/chatgpt-4-a-reussi-un-test-de-turing-faut-il-sinquieter/). Cependant, l'humain réalise souvent de façon plus précise les tâches du NLP (au moins les spécialistes du langage) et [il n'existe pas de solution intégrée pour réaliser toutes les tâches logiques du NLP](https://theconversation.com/beau-parleur-comme-une-ia-196084). Un arbitrage récurrent est alors de mettre en rapport les volumes et vitesses des opérations à réaliser face aux erreurs engendrées.

# 2. Le langage et les différentes approches du NLP

Afin de présenter le text mining et le NLP, il convient d'abord de se constituer le *kit minimum de culture générale* sur le domaine afin de comprendre les principales approches du domaine (et leur complémentarité). On en distingue souvent deux : l'approche dite **symbolique** (ou par expertise) et celle dite **fréquentiste** (ou linguistique de corpus). 

Dans cette présentation, on utilisera en majorité des modèles issus de l'approche fréquentiste, qui s'est particulièrement développée avec l'informatique moderne. Cependant, on va commencer par présenter quelques éléments de l'approche symbolique à travers des termes linguistiques au centre des "systèmes experts".... et utiles pour la suite. 

## 2.1 Linguistique et systèmes experts

L'approche symbolique est inspirée du modèle cognitif fonctionnel de l'esprit humain. On peut la qualifier de **méthode déductive** : elle repose sur les connaissances de la grammaire formelle et du formalisme logique. Les éléments suivants, issus de Tellier (2012), présentent la façon dont la linguistique est au centre de ces systèmes experts. 

Les premiers concepts utiles dans cette aproche ont été introduits par le linguiste Ferdinand de Saussure (1857-1913). Selon ce dernier, le **langage** est :

> la construction sociale d'un système de signes, 

où un **signe** est défini comme :  

> l'association arbitraire entre un *signifiant* (l'image acoustique d'un mot) et un *signifié* (représentation mentale d'une chose).

Dans ce cadre, les signes prennent du sens par les rapports qu'ils entretiennent les uns avec les autres dans le langage. Il apparaît alors deux niveaux dans *l'analyse linguistique* qui sont intégrés dans les systèmes experts :

- *l'analyse syntaxique* : étude de l'articulation des signes dans le langage, et


- *l'analyse sémantique* : étude du signifiant des signes (individuellement ou regroupés en proposition).

André Martinet (1908-1999) identifie plus tard la caractéristique (ou spécificité) des langages naturels humains, la **double articulation** (dont seule la seconde est étudiée par la suite) : 

- une première articulation regroupe les *phonèmes* (sons élémentaires des mots dans une langue donnée : [36 en français](https://www.ac-caen.fr/dsden50/circo/mortain/IMG/pdf/3_phonemes_francais.pdf)) dans des mots ou des *morphèmes* (plus petite unité du langage ayant un sens, dont les préfixes, suffixes, etc.).


- Une seconde articulation permet la combinaison des mots et morphèmes (syntaxe) en vue de donner du sens au contenu (sémantique). 

Enfin, pour comprendre le sens d'un document donné, il est peut être utile de prendre en compte l'objectif de son auteur. Dans un contexte de communication, Roman Jakobson (1896-1982) a déterminé **6 fonctions au langage** :

- expressive : le locuteur exprime un sentiment ou une idée,

- conative :  générer une action sur le destinataire (donner un ordre, etc.),

- référentielle : informer sur le monde extérieur,

- phatique: vérifier le bon fonctionnement de la communication ("allo", etc.),

- poétique : mettre l'accent sur la forme du message plus que sur son contenu informationnel, et

- métalinguistique : parler du langage par le langage (comme on le fait ici).

La figure suivante présente la relation entre les différentes strates du langage et les tâches classiques du text mining et du NLP. Les applications de la couche inférieure ne sont pas étudiées ici tandis que celles non encore présentées le sont dans cette introduction ou les autres notebooks. 

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Articulations du langage et tâches du NLP (Vajjala et al., 2020, p.9)</b><br>
  <img src="img/lang-artic_NLP-tasks.png" style="margin-top:0.5cm;margin-bottom:0cm;width:70%"> </div>
<div align="center" style="margin-top:0.1cm"> RQ : Les lexèmes sont des morphèmes ayant un sens par eux-mêmes. </div>
</figure>


Mais les "systèmes experts" tels qu'utilisé aujourd'hui se sont développés avec l'apparition de la *sémantique formelle* (ou [linguistique informatique](https://fr.wikipedia.org/wiki/Linguistique_informatique)) durant les années 1970. Ce champs d'analyse vise à reproduire les différentes étapes menant à la compréhension d'un énoncé tel que le fait l'esprit humain, et de formaliser des raisonnements. Cette approche repose notamment sur les travaux du linguiste Noam Chomsky (né en 1928) qui a participé à la création de ce qu'on appelle la [grammaire non contextuelle](https://fr.wikipedia.org/wiki/Grammaire_non_contextuelle).

Pour décrire simplement les éléments qui nous serviront plus loin, on utilise les trois figures suivantes. Elles représentent de façon simplifiée la chaîne de traitement du langage par l'esprit humain (modèle cognitif fonctionnel). Dans la première figure, **les cercles représentent des données** et **les rectangles des traitements de l'information**, dont des analyses (en entrée) et synthèses (en sortie) lexicales, syntaxiques et sémantiques. Les deux figures suivantes permettent de détailler ces étapes de traitement du langage de l'énoncé à la nouvelle phrase en réponse.

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Chaîne de traitements de compréhension du langage (Tellier, 2012, p.12)</b> <br>
  <img src="img/human-spirit.png" style="margin-top:0.5cm;margin-bottom:0cm;width:80%"> </div>
</figure>

Par exemple, pour la phrase "*un chat noir dort*", **l'analyse lexicale** peut être vue comme la détermination des unités lexicales (basée sur les espaces entre les mots) et la labélisation de leurs fonctions grammaticales (respectivement déterminant, nom, adjectif et verbe intransitif). Ensuite, **l'analyse syntaxique** correspond à la mise en relation de ces mots dans une combinaison logique (ici par la structuration en un groupe nominal) :
<figure >
    <div align="center" style="margin-top:0.5cm">
  <img src="img/human-spirit_syntax.png" style="margin-top:0.5cm;margin-bottom:0cm;width:30%">
    <div align="center" style="margin-top:0.5cm">
</figure>


**L'analyse sémantique** "limite" enfin la quantité d'information traitée en entrée en passant au niveau du signifiant de la proposition. La figure suivante reformule en ce sens la phrase en "sujet" et "prédicat" ([au sens linguistique](https://www.cnrtl.fr/definition/pr%C3%A9dicat) : c'est le terme qui affirme ou nie quelque chose du sujet, dans un énoncé où justement on peut identifier ce dont on parle). 
<figure>
    <div align="center" style="margin-top:0.5cm">
  <img src="img/human-spirit_semantic.png" style="margin-top:0.5cm;margin-bottom:0cm;width:30%">
    </div>
</figure>


Ces traitements sont effectués en sens inverse à la sortie dans la première figure, afin de formuler la réponse en langage naturel. Entre les deux il est fait référence à la mémoire. C'est à ce niveau qu'entre en jeu le "sens commun" évoqué plus haut : les réponses aux propositions "le chien a mordu le cycliste" et "le cycliste a mordu le chien" ne seront probablement pas les mêmes.

Cet exemple permet d'illustrer l'approche symbolique du NLP qui définit un ensemble de règles et de procédures à partir d'une expertise du langage. Elles sont ensuite appliquées aux tâches visées. Par exemple, ce type de règles expertes permet aux packages `VADER` (voir le [Github du package](https://github.com/cjhutto/vaderSentiment)) et `Textblob` (voir le [Github du package](https://github.com/sloria/TextBlob)) d'étudier le caractère positif ou négatif d'un avis rédigé en langage naturel : des mots sont chargés d'une valeur positive ou négative, et la syntaxe de l'avis est exploitée pour éventuellement inverser ces valeurs en cas de négation.

Cependant, cette approche présente des **problèmes majeurs** :

- faible degré de généralisation : difficulté d'une modélisation logique d'une langue en intégrant l'ensemble des cas possibles,


- rigidité : une langue peut évoluer rapidement et varier fortement selon les contextes d'utilsation (ex : réseaux sociaux),


- difficulté de transposition des règles entre différentes langues (objet de la linguistique comparative) : recherche complexe de points communs entre différentes langues (ou [*universaux linguistiques*](https://fr.wikipedia.org/wiki/Universaux_linguistiques)).

RQ : l'approche fréquentiste utilise souvent des briques issues de celle-ci : classification à partir de lexiques prédéfinis (dont l'élimination des mots vides de sens ou *stopwords*), extraction des entités nommées utilisant la syntaxe, recours à des bases de données de relations entre les mots (ex : [Wordnet](https://wordnet.princeton.edu/), qui peut être [visualisée en anglais ici](https://www.visual-thesaurus.com/wordnet.php)), etc. 

## 2.2 L'approche fréquentiste

Depuis les années 1990, la croissance de la puissance calculatoire informatique et des capacités de stockage ainsi que l'accès aux données en ligne ont engendré une évolution de l'analyse du langage naturel : vers une **approche inférentielle (ou statistique)**. Il s'agit donc d'une approche **inductive** : elle repose sur l'entrainement d'un algorithme (régression logistique ou autre) à partir des exemples d'un **corpus** (ensemble de documents textuels).

Un avantage immédiat de cette approche est qu'elle repose sur une dimension empirique lui permettant d'intégrer d'office  une partie du "sens commun" issue du corpus (en particulier si on modélise les probabilités conditionnelles dans les séquences de mots) : "*le chien a mordu le cycliste*" sera plus "probable" que la phrase inversée.

Par ailleurs, relativement à la méthode symbolique, le passage d'une langue à une autre est facilité car il peut "suffire" de passer d'un corpus dans une langue à un corpus dans l'autre et de renouveler l'entrainement initial.

De façon générale, l'approche fréquentiste vise à estimer, à partir d'exemples labélisés ou non (cf. la section suivante), les paramètres d'un modèle choisi pour une ou plusieurs tâches de NLP. 

Cependant, cette approche présente des **problèmes majeurs** :

- difficulté de construire des corpus annotés et adaptés à une tâche visée : il est parfois possible de bénéficier d'algorithmes pré-entrainés (particulièrement sous Python), mais il est souvent nécessaire d'effectuer un entrainement reposant sur des annotations potentiellement complexes à réaliser. De plus, le modèle obtenu peut avoir une mauvaise capacité prédictive si le corpus est de mauvaise qualité ("**Garbage In, Garbage Out**") ou s'il est inadapté à la tâche visée (ex : vocabulaires très différents entre le corpus et les textes à analyser).


- pas de prise en compte des étapes logiques de la construction du langage : l'approche symbolique (syntaxe + sémantique) doit parfois être intégrée dans les étapes d'une tâche à réaliser,


- difficulté d'interprétation de certains modèles (boîtes noires) : par exemple, les réseaux de neurones sont difficiles à interpréter lorsque le nombre de neurones et de couches augmentent ([comme illustré sur ce site](https://playground.tensorflow.org/)).

RQ : Si on veut se figurer l'équivalent de l'approche par expertise qui a donné lieu à Wordnet, on peut regarder le travail de l'équipe NLP de l'Université de Leipzig qui a décomposé les mots de plusieurs corpus dans plusieurs langues et réalisé des [graphes égocentrés basés sur leurs coocurrences](https://corpora.uni-leipzig.de/).

# 3. La modélisation pour l'inférence statistique

Pour appréhender simplement l'approche fréquentiste du NLP, et ce en quoi consiste un modèle en matière de données textuelles, on peut commencer par voir un texte comme un sac de mots (**Bags Of Words**) ou de séquences de $n$-mots (**Bags Of N-grams**). On peut alors chercher à étudier la distribution, au sens de loi de variables aléatoires, de ces mots ou séquences de mots. 

Une des premières étapes à réaliser est alors :

> la **vectorisation des textes** sur la base des mots contenus ou, dans des méthodes plus récentes, sur la base des mots et de leur contexte.

La figure suivante illustre une manière binaire de vectoriser un texte (dite encodage "*one-hot*") dans l'approche "Bag of words" (i.e. sans tenir compte du contexte).
<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Vectorisation par la méthode d'encodage binaire, dite "one-hot" (Bengfort et al., 2018, p.60)</b> <br>
  <img src="img/vect_one-hot.png" style="margin-top:0.5cm;margin-bottom:0cm;width:80%"></div>
</figure>



Dans la figure ci-dessus, le vecteur obtenu pour un document est construit à partir du vocabulaire de l'ensemble du corpus (composé ici de 18 éléments trouvés dans 3 documents). Pour obtenir le vecteur du premier document, chaque mot (dit "*token*" ou unité lexicale) est d'abord découpé à l'aide d'un **tokéniseur** qui est un algorithme d'analyse lexicale appliqué à une langue donnée (ici sans conserver la ponctuation). Les tokens sont ensuite **lemmatisés** (remplacés par la forme qui les a engendré : "*bats*" $\rightarrow$ "*bat*" dans la figure). Enfin, le lexique de l'ensemble du corpus définit la dimension du vecteur (18) et le vecteur de chaque document est rempli de 1 et 0 associés à chaque token qu'il contient ou non.

L'étape suivante dans l'approche fréquentiste consiste à utiliser les vecteurs issus des textes du corpus pour entrainer un algorithme, éventuellement en se reposant sur d'autres ayant été préalablement entrainés. Cet entrainement correspond à une optimisation soit à l'aide de labels préétablis (**apprentissage supervisé**) soit sans labels (**apprentissage non supervisé**), selon l'opération visée et les données disponibles. 

RQ : au centre de cette méthode on trouve donc le *machine learning* et le *deep learning*, mais on parle également de *transfer learning* lorsque l'on intègre un algorithme préentrainé au sein d'une étape additionnelle d'entrainement (ou adaptation) sur ses propres données.

## 3.1 Apprentissage supervisé

Un apprentissage supervisé est utilisé lorsqu'il existe un ensemble fini de valeurs (discrètes ou continues) à prédire et qu'on dispose d'exemples accompagnés de leur(s) vraie(s) valeur(s). L'entrainement consiste à minimiser les erreurs entre ces exemples (observations du "*training set*") et les prédictions du modèle choisi pour une tâche donnée (ex : tokénisation, classification de documents, probabilité de résiliation d'un client, appartenance et types des entités nommées dans un texte, etc.). On infère donc les caractéristiques des textes associées à des valeurs préexistantes et identifiées avant l'entrainement. 

La figure ci-dessous présente le cas le plus simple de la classification de mails avec un label binaire (spam ou ham).
<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Exemple d'apprentissage supervisé : la détection de spams (Géron, 2019, p.8)</b> <br>
  <img src="img/train_superv.png" style="margin-top:0.5cm;margin-bottom:0.5cm;width:65%"></div>
</figure>

Durant la phase opérationnelle (par le fournisseur d'un service de messagerie eléctronique par exemple), le déploiment du modèle commence par une étape de véctorisation des nouveaux documents identique à celle réalisée avant la phase d'entrainement, puis consiste en l'application des paramètres estimés du modèle pour leur asocier une (des) valeur(s).

## 3.2 Apprentissage non supervisé

Par opposition, un apprentissage non supervisé est réalisé sans qu'il soit indiqué de valeurs vraies ou fausses au moment de l'entrainement (ex : création de clusters des textes, topic modeling, etc.).  Dans le cas du "*clustering*", illustré ci-dessous, l'entrainement vise à regrouper des documents en se basant sur un "degré de similarité" des vecteurs des textes.
<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Exemple d'apprentissage non supervisé : clustering (Géron, 2019, p.10 et 11)</b> <br>
  <img src="img/train_unsuperv.png" style="margin-top:0.5cm;margin-bottom:0cm;width:90%;float:center"> </div>

# 4. Pipeline du text mining et du NLP

Quelle que soit l'approche retenue, il convient de toujours bien définir la tâche à réaliser sur les textes à traiter, ainsi que la séquence (dite "*pipeline*") des étapes intermédiaires nécessaires. 

La figure suivante illustre le pipeline habituel commençant par l'acuisition de données et intégrant des boucles d'amélioration avant la phase opérationnelle ("deployment") ou de mise à jour en cas d'évolution notable des nouveaux documents à traiter (observable durant la phase de "monitoring"). Jusque là, on a principalement parlé du "pre-pocessing" et de la modélisation.

<figure>
    <div align="center" style="margin-top:0.5cm"> <b> Pipeline générique de la fouille de texte (Vajjala et al., 2020, p.38)</b> <br>
  <img src="img/pipeline-generic.png" style="margin-top:0.5cm;margin-bottom:0cm;width:70%"></div>
</figure>

Avant tout, notons que la plupart des étapes dépendent de la tâche à réaliser, voire des algorithmes utilisés durant la modélisation. De plus, dans certains cas, la modélisation peut simplement correspondre à l'application  à nos données d'un algorithme pré-entrainé et disponible sur des plateformes mettant à disposition des API comme [Hugging Face](https://huggingface.co) ou [OpenAI](https://platform.openai.com/).

- **L'acquisition de données** : étape essentiellement reliée au contexte du projet et aux données dont on dispose (traitement de mails, exploitation d'un chatbot, traduction automatique d'un site, extraction d'information à partir d'avis d'utilisateurs, etc.). On peut également utiliser les méthodes du webscraping et/ou des bases de données disponibles en ligne (via [le moteur dédié de Google](https://datasetsearch.research.google.com/) et sites spécialisés, dont [Hugging Face](https://huggingface.co/datasets) ou la [page Github de Nicolas Iderhoff](https://github.com/niderhoff/nlp-datasets)). 


- **Nettoyage du texte** : cette étape vise à récupérer du texte au bon format depuis les fichiers à exploiter. Pour un fichier `txt`, cela se rapporte surtout à la détermination et/ou modification de l'encodage (en général vers [le standard `utf-8`](https://fr.wikipedia.org/wiki/UTF-8)) par exemple avec Notepad++. Pour les fichiers `html`, des packages de *parsing* des balises [comme `BeautifulSoup`](https://www.crummy.com/software/BeautifulSoup/) sont utiles. Des solutions s'attaquent [plus ou moins bien aux `pdf`](https://towardsdatascience.com/how-to-extract-text-from-pdf-245482a96de7) ou autre formats.


- **Pre-processing** : cette étape regroupe l'ensemble des opérations les plus spécifiques au text mining et au NLP, au sens des deux approches vues dans la section précédente. Il s'agit de transformer les textes encore sous forme brute (chaîne de caractères contenant tout le texte d'un document) en des tokens (chaînes de caractères correspondant à des mots, de la ponctuation ou autre) et qui sont souvent normalisés (corrections orthographiques, racinisations, etc.). Certaines informations syntaxiques ou sémantiques peuvent être utilisées à cette étape. 


- **Feature Engineering** (ingénierie des données) :  les documents sont vectorisés à cette étape. Il s'agit donc de la mise en forme des observations pour les rendre compatibles avec le ou les algorithmes de modélisation utilisés. Il est souvent utile de réaliser à ce moment des statistiques descriptives, réduction de dimension, etc. 


- **Modélisation** : si on entraine (ou adapte) un ou plusieurs algorithmes nécessaires à la tâche visée, on découpe tout d'abord les documents en des jeux d'entrainement, de validation et de test. On peut ensuite adopter un mécanisme de vote entre différents algorithmes (efficacité souvent supérieure au meilleur d'entre eux).


- **Evaluation** : cette étape permet de s'assurer que les performances du modèle sont satisfaisantes selon un critère défini à l'avance (sur l'échantillon de test). L'entrainement peut inclure un retour au pre-processing (pour amélioration) mais ne doit pas avoir reposé sur le test-set (uniquement sur le ou les écantillons de validation). Les scores de référence pour les opérations du NLP sont évoqués à la fin du prochain notebook ([voir par exemple cette présentation complète, dont le score BLEU utilisé pour la traduction](https://huggingface.co/metrics)).


- **Déploiement, monitoring et mise à jour du modèle** : le déploiement consiste à appliquer le meilleur modèle trouvé  à de nouveaux textes. Le monitoring et la mise à jour du modèle correspondent à la gestion de l'évolution de la tâche réalisée dans le temps ou plus généralement sur des documents susceptibles de ne pas correspondre à ceux utilisés pour l'entrainement.

# 5. Références

## 5.1 Cours, livres et articles 

Bengfort B., Bilbro B. et Ojeda T. (2018), "Applied Text Analysis with Python: Enabling Language-Aware Data Products with Machine Learning", Ed. O'Reilly.

Géron A. (2019), "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow: Concepts, Tools, and Techniques to
Build Intelligent Systems", 2nde édition, Ed. O'Reilly. (existe en version française en deux volumes)

Saffi S. (2005), "Les universaux linguistiques", Cahiers d’études romanes, 14, pp. 47-82. (disponible en ligne : https://journals.openedition.org/etudesromanes/2424)

Fabian Suchanek F. et Varoquaux G. (2022), "Beau parleur comme une IA", The Conversation. (disponible en ligne : https://theconversation.com/beau-parleur-comme-une-ia-196084)

Rakotomalala, R. (2017), Matériel du cours "NLP et Web Mining". (disponible en ligne : https://cours-machine-learning.blogspot.com/p/nlp-web-mining.html)

Tellier I. (2012), "Introduction au TALN et à l'ingénierie linguistique", cours à l'Université de Lille 3. (disponible en ligne : http://www.tal.univ-paris3.fr/travaux-etudiants/LZO21-2011-2012/Charlotte-Faure/images/livre.pdf)
 
Vajjala S., Majumder B., Gupta A. et Surana H. (2020), "Practical Natural Language Processing: A Comprehensive Guide to Building Real-World NLP Systems", Ed. O'Reilly.


## 5.2 Sites Internet 

Alexa : https://developer.amazon.com/fr-FR/alexa

Académie de Caen : https://www.ac-caen.fr/dsden50/circo/mortain/IMG/pdf/3_phonemes_francais.pdf

BeautifulSoup : https://www.crummy.com/software/BeautifulSoup/

CNRTL : https://www.cnrtl.fr/definition/pr%C3%A9dicat

DeepL : https://www.deepl.com/translator

Etalab-IA : https://www.data.gouv.fr/fr/reuses/modele-de-questions-reponses-francophone/ 

Github :
-  Nicolas Iderhoff : https://github.com/niderhoff/nlp-datasets
- VADER : https://github.com/cjhutto/vaderSentiment 
- Textblob  : https://github.com/sloria/TextBlob/

Google Colab : https://colab.research.google.com/

Google Translate : https://translate.google.fr/ 

Google Data set search : https://datasetsearch.research.google.com/

Hugging Face : https://huggingface.co

OpenAI (plateforme) : https://platform.openai.com/

Tensorflow : https://playground.tensorflow.org/

Towards Data Science (Medium) : https://towardsdatascience.com/how-to-extract-text-from-pdf-245482a96de7 

Université de Leipzig (partie graphs multilingues de mots) : https://corpora.uni-leipzig.de/

Visual Thesauring : https://www.visual-thesaurus.com/wordnet.php

Wikipedia : 

- Ambiguïté et désambiguïsation : https://fr.wikipedia.org/wiki/Ambigu%C3%AFt%C3%A9

- Grammaire non contextuelle : https://fr.wikipedia.org/wiki/Grammaire_non_contextuelle

- Linguistique informatique : https://fr.wikipedia.org/wiki/Linguistique_informatique

- Test de Turing : https://fr.wikipedia.org/wiki/Test_de_Turing

- Universaux linguistiques : https://fr.wikipedia.org/wiki/Universaux_linguistiques

- utf-8(encodage) : https://fr.wikipedia.org/wiki/UTF-8

Wordnet : https://wordnet.princeton.edu/

